In [ ]:
# ============================================================
# TOKENIZATION IN NLP
# RE, BPE, WordPiece, Unigram LM, SentencePiece
# With English, Hindi, Sanskrit examples
# ============================================================

from collections import Counter, defaultdict
import math

# ------------------------------------------------------------
# 0. DATA
# ------------------------------------------------------------
train_en = [
    "low lower lowest",
    "new newer newest",
    "wide wider widest",
    "fast faster fastest",
    "smart smarter smartest"
]

train_hi = [
    "घर घरों घराना",
    "बड़ा बड़े बड़ी"
]

train_sa = [
    "रामः रामौ रामाः",
    "गच्छति गच्छतः गच्छन्ति"
]

test_data = [
    "slowest smarter",
    "घरों घराना",
    "रामाः गच्छन्ति"
]

print("TRAIN DATA")
print(train_en)
print(train_hi)
print(train_sa)
print("\nTEST DATA")
print(test_data)

# ------------------------------------------------------------
# 1. REGEX / RULE-BASED TOKENIZER
# ------------------------------------------------------------
class RegexTokenizer:
    def __init__(self):
        self.suffixes = ["est","er","ों","ाना","आः","न्ति"]

    def tokenize_word(self, word):
        for suf in self.suffixes:
            if word.endswith(suf) and len(word) > len(suf):
                return [word[:-len(suf)], suf]
        return [word]

    def tokenize(self, sent):
        out = []
        for w in sent.split():
            out.extend(self.tokenize_word(w))
        return out

re_tok = RegexTokenizer()

print("\n=== REGEX TOKENIZATION ===")
for s in test_data:
    print(s, "->", re_tok.tokenize(s))

# ------------------------------------------------------------
# 2. BPE TOKENIZER (from scratch)
# ------------------------------------------------------------
class BPE:
    def __init__(self, num_merges=30):
        self.num_merges = num_merges
        self.merges = []

    def get_stats(self, vocab):
        pairs = defaultdict(int)
        for word,freq in vocab.items():
            toks = word.split()
            for i in range(len(toks)-1):
                pairs[(toks[i],toks[i+1])] += freq
        return pairs

    def merge_vocab(self, pair, vocab):
        new_vocab = {}
        bigram = " ".join(pair)
        rep = "".join(pair)
        for w in vocab:
            new_vocab[w.replace(bigram, rep)] = vocab[w]
        return new_vocab

    def train(self, corpus):
        vocab = Counter()
        for s in corpus:
            for w in s.split():
                vocab[" ".join(list(w))] += 1

        for _ in range(self.num_merges):
            pairs = self.get_stats(vocab)
            if not pairs: break
            best = max(pairs, key=pairs.get)
            self.merges.append(best)
            vocab = self.merge_vocab(best, vocab)

    def tokenize_word(self, word):
        toks = list(word)
        for a,b in self.merges:
            i = 0
            while i < len(toks)-1:
                if toks[i]==a and toks[i+1]==b:
                    toks[i:i+2] = [a+b]
                else:
                    i += 1
        return toks

# train on all languages together
bpe = BPE(40)
bpe.train(train_en + train_hi + train_sa)

print("\n=== BPE MERGE TREE (first 15 merges) ===")
for i,m in enumerate(bpe.merges[:15]):
    print(i+1, ":", m[0], "+", m[1], "->", m[0]+m[1])

print("\n=== BPE TOKENIZATION ===")
for s in test_data:
    print(s, "->", [bpe.tokenize_word(w) for w in s.split()])

# ------------------------------------------------------------
# 3. WORDPIECE (LIKELIHOOD-BASED BPE)
# ------------------------------------------------------------
class WordPiece(BPE):
    def token_counts(self, vocab):
        c = Counter()
        for w,f in vocab.items():
            for t in w.split():
                c[t] += f
        return c

    def log_likelihood(self, vocab):
        c = self.token_counts(vocab)
        total = sum(c.values())
        ll = 0
        for t,f in c.items():
            p = f/total
            ll += f * math.log(p)
        return ll

    def score_merge(self, vocab, pair):
        new_vocab = self.merge_vocab(pair, vocab)
        return self.log_likelihood(new_vocab) - self.log_likelihood(vocab)

    def train(self, corpus):
        vocab = Counter()
        for s in corpus:
            for w in s.split():
                vocab[" ".join(list(w))] += 1

        for _ in range(self.num_merges):
            pairs = self.get_stats(vocab)
            if not pairs: break
            best = max(pairs, key=lambda p:self.score_merge(vocab,p))
            self.merges.append(best)
            vocab = self.merge_vocab(best, vocab)

wp = WordPiece(30)
wp.train(train_en + train_hi + train_sa)

print("\n=== WORDPIECE MERGES (first 10) ===")
print(wp.merges[:10])

print("\n=== WORDPIECE TOKENIZATION ===")
for s in test_data:
    print(s, "->", [wp.tokenize_word(w) for w in s.split()])

# ------------------------------------------------------------
# 4. UNIGRAM LANGUAGE MODEL TOKENIZER
# ------------------------------------------------------------
class UnigramLM:
    def __init__(self, vocab_size=200):
        self.vocab_size = vocab_size
        self.probs = {}

    def all_substrings(self, w):
        subs=set()
        for i in range(len(w)):
            for j in range(i+1,len(w)+1):
                subs.add(w[i:j])
        return subs

    def train(self, corpus):
        cand = Counter()
        for s in corpus:
            for w in s.split():
                for sub in self.all_substrings(w):
                    cand[sub]+=1

        top = dict(cand.most_common(self.vocab_size))
        total = sum(top.values())
        self.probs = {k:v/total for k,v in top.items()}

    def tokenize(self, word):
        n=len(word)
        dp=[(-1e9,[])]*(n+1)
        dp[0]=(0,[])
        for i in range(n):
            if dp[i][0]<-1e8: continue
            for j in range(i+1,n+1):
                sub = word[i:j]
                if sub in self.probs:
                    sc = dp[i][0] + math.log(self.probs[sub])
                    if sc > dp[j][0]:
                        dp[j]=(sc, dp[i][1]+[sub])
        return dp[n]

uni = UnigramLM(300)
uni.train(train_en + train_hi + train_sa)

print("\n=== UNIGRAM TOKENIZATION ===")
for s in test_data:
    print(s)
    for w in s.split():
        sc,toks = uni.tokenize(w)
        print(" ",w,"->",toks,"logP=",round(sc,3))

# ------------------------------------------------------------
# 5. SENTENCEPIECE STYLE (SPACE AS TOKEN)
# ------------------------------------------------------------
def sp_prep(sent):
    return "▁" + sent.replace(" ", " ▁")

sp_train = [sp_prep(s) for s in train_en+train_hi+train_sa]
sp_test  = [sp_prep(s) for s in test_data]

sp_uni = UnigramLM(400)
sp_uni.train(sp_train)

print("\n=== SENTENCEPIECE TOKENIZATION ===")
for s in sp_test:
    sc,toks = sp_uni.tokenize(s)
    print(s,"->",toks)

# ------------------------------------------------------------
# 6. LIKELIHOOD TABLE (UNIGRAM)
# ------------------------------------------------------------
print("\n=== LIKELIHOOD TABLE (SAMPLE TOKENS) ===")
sample_tokens = ["राम","आः","गच्छ","न्ति","घर","ओं"]
freqs = Counter()

for s in train_sa + train_hi:
    for w in s.split():
        for t in uni.tokenize(w)[1]:
            freqs[t]+=1

total = sum(freqs.values())

print("Token | Freq | Prob | logP")
print("----------------------------")
for t in sample_tokens:
    f = freqs[t]
    if f>0:
        p = f/total
        print(f"{t:6} | {f:4} | {p:5.3f} | {math.log(p):6.3f}")

# ------------------------------------------------------------
# 7. CORPUS LOG-LIKELIHOOD (UNIGRAM)
# ------------------------------------------------------------
def corpus_ll(model, corpus):
    ll=0
    for s in corpus:
        for w in s.split():
            ll += model.tokenize(w)[0]
    return ll

print("\n=== CORPUS LOG-LIKELIHOOD (UNIGRAM) ===")
print("LL =", round(corpus_ll(uni, train_en+train_hi+train_sa),3))

# ------------------------------------------------------------
# 8. OOV HANDLING DEMO
# ------------------------------------------------------------
oov_words = ["slowest","घरवाला","अज्ञातशब्द","नवीनतम"]

print("\n=== OOV HANDLING ===")
for w in oov_words:
    print("\nWORD:",w)
    print("Regex     :", re_tok.tokenize_word(w))
    print("BPE       :", bpe.tokenize_word(w))
    print("WordPiece :", wp.tokenize_word(w))
    print("Unigram   :", uni.tokenize(w)[1])

# ------------------------------------------------------------
# 9. UTF-8 BYTE ANALYSIS
# ------------------------------------------------------------
def bytes_utf8(s):
    return len(s.encode("utf-8"))

print("\n=== UTF-8 BYTE LENGTH ===")
chars = ["a","क","घर","रामः","🙂"]
for c in chars:
    print(c,"->",bytes_utf8(c),"bytes")

# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------
print("\n=== SUMMARY ===")
print("Regex        : rule-based, no probabilities")
print("BPE          : frequency-based merges")
print("WordPiece    : likelihood-based merges")
print("Unigram LM   : full probabilistic tokenizer")
print("SentencePiece: framework, handles spaces & multilingual text")

TRAIN DATA
['low lower lowest', 'new newer newest', 'wide wider widest', 'fast faster fastest', 'smart smarter smartest']
['घर घरों घराना', 'बड़ा बड़े बड़ी']
['रामः रामौ रामाः', 'गच्छति गच्छतः गच्छन्ति']

TEST DATA
['slowest smarter', 'घरों घराना', 'रामाः गच्छन्ति']

=== REGEX TOKENIZATION ===
slowest smarter -> ['slow', 'est', 'smart', 'er']
घरों घराना -> ['घर', 'ों', 'घर', 'ाना']
रामाः गच्छन्ति -> ['रामाः', 'गच्छ', 'न्ति']

=== BPE MERGE TREE (first 15 merges) ===
1 : s + t -> st
2 : e + r -> er
3 : e + st -> est
4 : र + ा -> रा
5 : l + o -> lo
6 : lo + w -> low
7 : n + e -> ne
8 : ne + w -> new
9 : w + i -> wi
10 : wi + d -> wid
11 : f + a -> fa
12 : fa + st -> fast
13 : s + m -> sm
14 : sm + a -> sma
15 : sma + r -> smar

=== BPE TOKENIZATION ===
slowest smarter -> [['s', 'lowest'], ['smarter']]
घरों घराना -> [['घरों'], ['घ', 'रा', 'न', 'ा']]
रामाः गच्छन्ति -> [['राम', 'ा', 'ः'], ['गच्छ', 'न', '्', 'त', 'ि']]

=== WORDPIECE MERGES (first 10) ===
[('s', 't'), ('i', 'd'), ('l', 'o'),